# 02 — Entanglement: states that aren't made of parts

## What you will learn

Notebook 01 built one qubit and then two. A two-qubit state was a $(2,2)$ array,
and the examples so far were all built by doing something to the first qubit and
something to the second. This notebook is about the states that *cannot* be built
that way — and there turn out to be a lot of them.

By the end you will know:

- what a **product state** is, and what it looks like when a system really is made
  of independent parts;
- what a **Bell state** is, and the two-line piece of algebra that proves it is not
  a product of anything — this is the definition of **entanglement**, and it is the
  heart of the notebook;
- what a **density matrix** is, what the **partial trace** does, and why one half of
  a Bell pair is *maximally uncertain* even though the pair as a whole is perfectly
  definite;
- how to put a number on it: the **Schmidt decomposition** and the **entanglement
  entropy**, which is exactly $1$ bit for a Bell pair;
- what measuring one half does to the other half, and what the three-qubit **GHZ**
  state's all-or-nothing correlations look like;
- how entanglement grows as a circuit runs;
- and why a `Qubit` object in this library deliberately has no `.state` attribute —
  a design decision that is really a statement about physics.

A framing worth holding onto from the start. It is tempting to read entanglement as
a strange extra ingredient sprinkled onto an otherwise classical world. The
arithmetic points the other way. The generic state of several qubits is entangled;
product states are the rare, thin, special case. What needs explaining is not why
entanglement exists, but why the world we walk around in looks like it is made of
separate objects with their own properties. That question has an answer, and later
notebooks get to it. This one just establishes how unusual "made of parts" is.

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np

from qsim import Circuit, Qubit, viz
from qsim.errors import NoCloningError
from qsim.gates import CNOT, H, Ry

# Print small floats readably instead of in scientific notation.
np.set_printoptions(precision=3, suppress=True)


def bits(value: float) -> float:
    """Round an entropy for display. The `+ 0.0` turns floating-point -0.0 into 0.0."""
    return round(value, 12) + 0.0

## 1. What "made of parts" looks like

Start with two qubits, both in $|0\rangle$, and apply a Hadamard to each one
separately. Nothing here couples the qubits: `H(a)` touches only qubit `a`, `H(b)`
touches only qubit `b`. Whatever comes out is, by construction, "qubit `a` in some
state, and independently qubit `b` in some state".

Recall from notebook 01 that $H|0\rangle = \tfrac{1}{\sqrt 2}(|0\rangle + |1\rangle)$,
usually written $|{+}\rangle$. So we expect

$$|{+}\rangle \otimes |{+}\rangle
 = \tfrac{1}{\sqrt2}(|0\rangle + |1\rangle) \otimes \tfrac{1}{\sqrt2}(|0\rangle + |1\rangle)
 = \tfrac{1}{2}\big(|00\rangle + |01\rangle + |10\rangle + |11\rangle\big).$$

The tensor product $\otimes$ multiplies out exactly like an ordinary product of two
sums: every term on the left pairs with every term on the right, and the amplitudes
multiply.

In [ ]:
qc = Circuit(name="product", seed=1234)
a, b = qc.alloc_many(2)   # two fresh qubits, both |0>, handles into this circuit
H(a)
H(b)

qc.inspect.ket()

Four terms, each with amplitude $0.5$, exactly as the algebra predicted. Reading the
labels: the leftmost bit is qubit `a` and the rightmost is qubit `b`, because
**qubit 0 is the most significant bit** everywhere in qsim.

Squaring those amplitudes gives four equal probabilities of $0.25$. Drawn as bars:

In [ ]:
print(qc.inspect.probabilities())
fig = viz.amplitudes(qc)

Four equal bars. Now here is the question this whole notebook turns on: *is that
state made of two independent halves, or not?*

For this one we already know the answer — we built it out of halves. The library can
confirm it from the state alone, without knowing how it was made. `is_product([a])`
asks whether qubit `a` is independent of everything else in the circuit:

In [ ]:
print("a independent of b? ", qc.inspect.is_product([a]))
print("b independent of a? ", qc.inspect.is_product([b]))

Another way to see the same thing: each qubit has a **Bloch vector** of its own.
Notebook 01 introduced the Bloch sphere — every single-qubit state corresponds to a
point in a unit ball, with $|0\rangle$ at the north pole, $|1\rangle$ at the south,
and $|{+}\rangle$ and $|{-}\rangle$ at opposite points on the equator. A vector of
length exactly $1$ (a point *on* the sphere) means the qubit has a definite state of
its own.

In [ ]:
print("Bloch vector of a:", np.round(qc.inspect.bloch_vector(a), 6))
print("Bloch vector of b:", np.round(qc.inspect.bloch_vector(b), 6))
print("length of a's vector:", np.linalg.norm(qc.inspect.bloch_vector(a)))

fig = viz.bloch(qc, a)

Both qubits sit at $(1, 0, 0)$ — the $|{+}\rangle$ point on the equator — with vector
length $1$, give or take the $10^{-16}$ that double-precision arithmetic always leaves
behind. Each one has a state, that state is $|{+}\rangle$, and the joint state is
nothing more than the pair of them. This is what a system genuinely made of parts
looks like, and it is the baseline against which the next section should be read.

A state that factors like this is called a **product state**.

## 2. The Bell state, and the algebra that breaks

Now change one thing. Apply `H` to the first qubit as before, but instead of a
second independent `H`, apply a **CNOT**: a two-qubit gate that flips the target
qubit if and only if the control qubit is $1$. It is the standard entangling gate,
and notebook 01 met it as a piece of reversible classical logic.

Follow it on basis states. CNOT sends $|00\rangle \to |00\rangle$ (control is 0, do
nothing) and $|10\rangle \to |11\rangle$ (control is 1, flip the target). Gates are
linear, so a superposition goes term by term:

$$\text{CNOT}\ \tfrac{1}{\sqrt2}\big(|00\rangle + |10\rangle\big)
 = \tfrac{1}{\sqrt2}\big(|00\rangle + |11\rangle\big).$$

Two gates, and we are done. The result is called a **Bell state** (after John Bell,
who appears properly in notebook 03).

In [ ]:
bell = Circuit(name="bell", seed=1234)
p, q = bell.alloc_many(2)
H(p)
CNOT(p, q)

bell.inspect.ket()

In [ ]:
print(bell.inspect.probabilities())
fig = viz.amplitudes(bell)

Two bars, at $|00\rangle$ and $|11\rangle$, each with probability $0.5$. The states
$|01\rangle$ and $|10\rangle$ have amplitude exactly zero: the two qubits *always*
read the same value.

That much is not yet surprising — a coin flip copied onto two pieces of paper would
also always agree. What is surprising is what happens when you try to write this
state as "qubit `p` in some state, and qubit `q` in some state".

### Try to factor it

Suppose there were single-qubit states $\alpha|0\rangle + \beta|1\rangle$ and
$\gamma|0\rangle + \delta|1\rangle$ whose product is the Bell state. Multiplying out:

$$\big(\alpha|0\rangle + \beta|1\rangle\big) \otimes \big(\gamma|0\rangle + \delta|1\rangle\big)
= \alpha\gamma\,|00\rangle + \alpha\delta\,|01\rangle + \beta\gamma\,|10\rangle + \beta\delta\,|11\rangle.$$

For that to equal $\tfrac{1}{\sqrt2}|00\rangle + 0\,|01\rangle + 0\,|10\rangle + \tfrac{1}{\sqrt2}|11\rangle$,
all four coefficients must match:

$$\alpha\gamma = \tfrac{1}{\sqrt2}, \qquad
  \alpha\delta = 0, \qquad
  \beta\gamma = 0, \qquad
  \beta\delta = \tfrac{1}{\sqrt2}.$$

Now walk the consequences:

1. $\alpha\gamma = \tfrac{1}{\sqrt2} \neq 0$, so $\alpha \neq 0$ **and** $\gamma \neq 0$.
2. $\alpha\delta = 0$ with $\alpha \neq 0$ forces $\delta = 0$.
3. $\beta\gamma = 0$ with $\gamma \neq 0$ forces $\beta = 0$.
4. But then $\beta\delta = 0 \cdot 0 = 0$, and it was supposed to be $\tfrac{1}{\sqrt2}$.

Contradiction. There are no such $\alpha, \beta, \gamma, \delta$ — not "we could not
find them", not "they are hard to compute". They do not exist, over the complex
numbers, at all.

> **Definition.** A multi-qubit state is **entangled** when it cannot be written as a
> tensor product of states of its parts. That failure to factor *is* entanglement.
> There is nothing else to it.

Notice what this definition does *not* say. It says nothing about distance, or
speed, or information travelling anywhere. It is a statement about which vectors in
$\mathbb{C}^4$ are products of vectors in $\mathbb{C}^2$ — and most of them are not.
Counting real dimensions, after fixing normalization and discarding the unobservable
overall phase, two-qubit states form a 6-dimensional space while the product states
form a 4-dimensional family inside it. Pick a state at random and you will essentially
never land on the thin part.

### The same argument in one line of linear algebra

Since you know linear algebra, here is the compact version. Arrange the four
amplitudes into a $2 \times 2$ matrix $M$, where $M_{ij}$ is the amplitude of
$|ij\rangle$ — for a two-qubit state, that is literally the $(2,2)$ state array. Then

$$\text{the state is a product} \iff M = u\,v^{\mathsf T} \text{ for some vectors } u, v
 \iff \operatorname{rank} M \le 1 \iff \det M = 0.$$

The four scalar equations above are just "$M$ is an outer product", written out.

In [ ]:
# state_tensor() returns the state in its native shape (2,)*n. For two qubits that
# is already the matrix M above: M[i, j] is the amplitude of |ij>.
m_product = qc.inspect.state_tensor()
m_bell = bell.inspect.state_tensor()

print("M for the product state:\n", m_product.real)
print("det =", np.linalg.det(m_product).real, " rank =", np.linalg.matrix_rank(m_product))
print()
print("M for the Bell state:\n", m_bell.real)
print("det =", np.linalg.det(m_bell).real, " rank =", np.linalg.matrix_rank(m_bell))

Rank 1 versus rank 2. That single integer is the whole distinction, and section 4
turns it into a continuous measure by looking at the *singular values* of $M$ rather
than just counting the nonzero ones.

(We took the real part only for printing; both matrices happen to be real here. In
general the amplitudes are complex and the argument is unchanged.)

## 3. What does one half of a Bell pair look like?

The Bell state is a perfectly definite state of the pair. We know it exactly — there
is no ignorance anywhere in it. So: what is the state of qubit `p` alone?

The factoring argument already told us there is no answer of the form "some vector in
$\mathbb{C}^2$". But we still want to describe what someone holding only qubit `p`
would see, and for that a state vector is not a rich enough language. We need a
slightly bigger one.

### Density matrices

A **density matrix** $\rho$ is the generalization of a state vector that can also
describe a *statistical mixture*. Given a state vector $|\psi\rangle$, its density
matrix is the outer product $\rho = |\psi\rangle\langle\psi|$ — for one qubit, a
$2 \times 2$ Hermitian matrix with trace $1$. If instead you have a classical
probability $p_k$ of the system being in each of several states $|\psi_k\rangle$, the
density matrix is the weighted sum $\sum_k p_k |\psi_k\rangle\langle\psi_k|$.

Reading one:

- the **diagonal** entries are the probabilities of measuring each basis state;
- the **off-diagonal** entries are called **coherences**, and they are what remains
  of superposition. A genuine superposition has large coherences; a mere
  classical mixture — "it is $|0\rangle$ or $|1\rangle$, we just don't know which" —
  has none. This is the difference the diagonal alone cannot see, and it is the
  difference that interference acts on.

Two examples make the distinction concrete. Both have diagonal $(\tfrac12, \tfrac12)$:

$$|{+}\rangle\langle{+}| = \begin{pmatrix} 0.5 & 0.5 \\ 0.5 & 0.5\end{pmatrix}
\qquad\text{versus}\qquad
\tfrac12 I = \begin{pmatrix} 0.5 & 0 \\ 0 & 0.5\end{pmatrix}.$$

The first is the pure state $|{+}\rangle$ — a definite state that happens to give
50/50 answers to a $Z$ measurement. The second, called the **maximally mixed state**,
is a coin flip: no state at all, just ignorance. The right-hand one is what "I know
nothing about this qubit" looks like.

### The partial trace

The operation that goes from the joint state to the state of one part is the
**partial trace**: you average away everything you chose not to look at. Concretely,
you sum over the ignored qubits' indices,

$$\rho_A[i, i'] = \sum_j M[i, j]\, \overline{M[i', j]},$$

which in matrix form is just $\rho_A = M M^\dagger$ — and $M$ is the same matricized
state from section 2, with the kept qubits indexing rows and the ignored ones
indexing columns. `reduced_density_matrix` does exactly this.

Let us look at one qubit from each of our two states.

In [ ]:
rho_product = qc.inspect.reduced_density_matrix([a])      # a, from |+>|+>
rho_bell = bell.inspect.reduced_density_matrix([p])       # p, from the Bell pair

print("one qubit of the product state:\n", rho_product)
print("\none qubit of the Bell state:\n", rho_bell)

Look at what happened.

The product state's qubit came back as $\begin{pmatrix}0.5 & 0.5\\ 0.5 & 0.5\end{pmatrix}$
— the pure state $|{+}\rangle$, coherences and all. Nothing was lost by ignoring the
other qubit, because the other qubit was never involved.

The Bell pair's qubit came back as $\tfrac12 I$: the maximally mixed state. The
diagonal says 50/50, and the coherences are *gone*. Someone handed qubit `p` and
nothing else cannot distinguish it from a fair coin that has already been flipped.

This is the sharp version of "an entangled qubit has no state of its own". Not that
we are ignorant of its state — the pair's state is known perfectly — but that the
question "what is `p`'s state?" has no answer of that kind. Every bit of what is
going on lives in the *relationship* between the two qubits, and the partial trace,
by throwing the relationship away, is left holding nothing.

### The same thing on the Bloch sphere

The Bloch vector is read off the reduced density matrix, so it tells the same story
geometrically: pure states lie on the surface (length 1), mixed states strictly
inside, and the maximally mixed state sits at the exact centre.

In [ ]:
v = bell.inspect.bloch_vector(p)
print("Bloch vector of one half of a Bell pair:", np.round(v, 12))
print("its length:", float(np.linalg.norm(v)))

fig = viz.bloch(bell, p)

An arrow of length zero, sitting at the origin. There is no direction to point.

Hold the two pictures side by side. In section 1 the two-qubit state was fully
described by two arrows of length 1, one per qubit. Here the two-qubit state is just
as sharply defined — one specific unit vector in $\mathbb{C}^4$ — and yet both
per-qubit arrows have collapsed to a point. The information did not go missing; it
was never distributed to the parts in the first place. Classically that is not
possible: if you know a composite system perfectly, you know each of its pieces
perfectly. Quantum mechanics allows maximal knowledge of the whole together with
minimal knowledge of every part, and *that* is the structural novelty.

## 4. Putting a number on it

"Entangled or not" is a yes/no answer, and section 2 reduced it to "is $\operatorname{rank} M > 1$?".
But rank is a blunt instrument: a state can be barely entangled or maximally
entangled, and both have rank 2. We want a continuous measure, and the singular
values of $M$ provide it.

### The Schmidt decomposition

Take the singular value decomposition $M = U \Sigma V^\dagger$. Translated back into
state language, this says that **any** pure state of a bipartite system can be
written as a single sum

$$|\psi\rangle = \sum_i s_i\, |a_i\rangle \otimes |b_i\rangle$$

with orthonormal $|a_i\rangle$ on one side and orthonormal $|b_i\rangle$ on the other,
and non-negative $s_i$ — the singular values. This is the **Schmidt decomposition**,
and the $s_i$ are the **Schmidt coefficients**. Normalization forces
$\sum_i s_i^2 = 1$, so the $s_i^2$ are a probability distribution. They are also
exactly the eigenvalues of the reduced density matrix $\rho_A = MM^\dagger$, which is
a satisfying way to see that both sides of the cut are equally entangled: $MM^\dagger$
and $M^\dagger M$ have the same nonzero eigenvalues.

The point of the decomposition is that it is a *single* sum, not a double one. A
generic two-qubit state written in the computational basis needs four terms; the
Schmidt form needs at most two, and how many it actually needs — and how evenly the
weight is spread over them — is the entanglement.

- One nonzero coefficient ($s = (1, 0)$): a product state.
- Two equal coefficients ($s = (\tfrac{1}{\sqrt2}, \tfrac{1}{\sqrt2})$): maximally
  entangled.

In [ ]:
print("product state, Schmidt coefficients: ", qc.inspect.schmidt_spectrum([a]))
print("Bell state,    Schmidt coefficients: ", bell.inspect.schmidt_spectrum([p]))

$(1, 0)$ versus $(0.707, 0.707)$ — one term versus two equal terms. (The product
state's second coefficient prints as `0.` but is really something like $10^{-17}$;
that is floating-point noise, not physics.)

### Entanglement entropy

To collapse the spectrum into one number, apply the Shannon entropy to the
probability distribution $p_i = s_i^2$:

$$S = -\sum_i p_i \log_2 p_i \quad \text{bits}.$$

Shannon entropy measures how spread out a distribution is: it is $0$ when one
outcome is certain, and $\log_2 d$ when $d$ outcomes are equally likely. Applied to
the Schmidt spectrum it is called the **entanglement entropy**, and it is measured in
bits by default (pass `base=np.e` for nats).

For one qubit the spectrum has at most two entries, so $S$ runs from $0$ to $1$ bit.

In [ ]:
print("product state, entropy of qubit a:", bits(qc.inspect.entanglement_entropy([a])), "bits")
print("Bell state,    entropy of qubit p:", bits(bell.inspect.entanglement_entropy([p])), "bits")
print("Bell state, same thing in nats:   ",
      bits(bell.inspect.entanglement_entropy([p], base=np.e)), "nats  (= ln 2)")

Zero and one, on the nose. A Bell pair carries exactly one bit ("one ebit") of
entanglement, which is the most two qubits can share: knowing the pair perfectly
tells you nothing whatsoever about either half, and "nothing whatsoever about one
qubit" is precisely one bit of missing information.

Entanglement is not all-or-nothing, though. Replace the Hadamard with a rotation
`Ry(a, theta=θ)`, which takes $|0\rangle$ to $\cos(\theta/2)|0\rangle + \sin(\theta/2)|1\rangle$,
and then apply the CNOT. The result is $\cos(\theta/2)|00\rangle + \sin(\theta/2)|11\rangle$,
and $\theta$ dials the entanglement continuously.

In [ ]:
def tilted_pair(theta: float) -> Circuit:
    """cos(theta/2)|00> + sin(theta/2)|11>: a Bell pair with adjustable tilt."""
    c = Circuit(seed=0)
    x, y = c.alloc_many(2)
    Ry(x, theta=theta)     # angles are keyword-only throughout qsim
    CNOT(x, y)
    return c


print(f"{'theta':>10} {'Schmidt coefficients':>28} {'entropy (bits)':>16}")
for theta in [0.0, np.pi / 8, np.pi / 4, np.pi / 2, 3 * np.pi / 4, np.pi]:
    c = tilted_pair(theta)
    first = c.qubits[0]
    s = c.inspect.schmidt_spectrum([first])
    entropy = bits(c.inspect.entanglement_entropy([first]))
    print(f"{theta:10.4f} {str(np.round(s, 4)):>28} {entropy:16.4f}")

At $\theta = 0$ nothing happened and the state is $|00\rangle$: one Schmidt
coefficient, zero entropy. At $\theta = \pi$ the state is $|11\rangle$ — also a
product state, also zero. The maximum sits at $\theta = \pi/2$, where the two Schmidt
coefficients are equal and the entropy is exactly 1 bit; that is the Bell state again,
since $R_y(\pi/2)|0\rangle = |{+}\rangle$ exactly. Everywhere in between you get a
fraction of a bit — entanglement comes in a continuum, not in units.

One more useful number, which will matter in later notebooks: the **mutual
information** $I(A{:}B) = S(A) + S(B) - S(AB)$, the total correlation between two
groups of qubits. For a Bell pair, $S(A) = S(B) = 1$ and $S(AB) = 0$ (the pair as a
whole is pure), giving $2$ bits — twice what any classical pair of bits can manage,
which is a first quantitative hint that this correlation is of a different kind.

In [ ]:
mi = bits(bell.inspect.mutual_information([p], [q]))
print("mutual information of the Bell pair:", mi, "bits")

## 5. Measuring one half

So far nothing has been measured. The Bell state sits there, both qubits maximally
mixed. Now measure qubit `p`.

Measurement in qsim projects the **joint** state: the amplitudes of the branch that
did not happen are set to zero and the rest is renormalized. That single operation
updates both qubits at once, because there were never two separate descriptions to
update — there was one tensor.

The outcome is genuinely random, 50/50. To see both possibilities we build the same
circuit twice with different random seeds. `Circuit(seed=...)` fixes the random
number generator so a run can be repeated; the physics is unchanged, only the coin
flips are reproducible.

In [ ]:
def fresh_bell(seed: int) -> tuple[Circuit, Qubit, Qubit]:
    c = Circuit(name="bell", seed=seed)
    x, y = c.alloc_many(2)
    H(x)
    CNOT(x, y)
    return c, x, y


for seed in (1, 2):
    c, x, y = fresh_bell(seed)
    print(f"--- seed {seed} ---")
    print("  before:      ", c.inspect.ket(), "  entropy of x:",
          bits(c.inspect.entanglement_entropy([x])), "bits")
    outcome = c.measure(x)
    print("  measured x ->", outcome)
    print("  after:       ", c.inspect.ket(), "  entropy of x:",
          bits(c.inspect.entanglement_entropy([x])), "bits")
    print("  Bloch vector of y:", np.round(c.inspect.bloch_vector(y), 6))

Read the two blocks together.

With seed 1 the measurement returned $0$ and the state became $|00\rangle$; qubit `y`,
which was maximally mixed a moment ago, is now at the north pole of the Bloch sphere
— definitely $|0\rangle$, vector length 1. With seed 2 the measurement returned $1$
and everything went the other way: $|11\rangle$, and `y` sits at the south pole.

In both cases the entanglement entropy dropped to zero. The state after measurement
is a product state, so the entanglement is gone — spent, in a sense, on producing
the correlation.

It is worth being careful about what did and did not just happen. Nothing was sent
from `p` to `q`. If you average over the two outcomes with their probabilities,
$\tfrac12 |0\rangle\langle 0| + \tfrac12 |1\rangle\langle 1| = \tfrac12 I$, which is
exactly `q`'s reduced density matrix *before* the measurement. Someone holding `q`
alone, with no access to `p`'s result, sees precisely nothing change. The correlation
is only visible once the two results are brought together and compared — which
requires an ordinary classical message.

At which point a reasonable objection arrives: a pair of gloves in two boxes also
always agree, and nobody calls that spooky. What distinguishes entanglement from that
kind of pre-arranged classical correlation? That is exactly the question notebook 03
answers, and the answer is not "nothing".

A first sign that something more is going on: the correlation survives a change of
question. `expectation` gives the average value of a Pauli observable — the average
result of measuring it many times, always between $-1$ and $+1$.

In [ ]:
b2, x2, y2 = fresh_bell(7)
for pauli, reading in [("ZZ", "measure both in the Z basis: always agree"),
                       ("XX", "measure both in the X basis: also always agree"),
                       ("YY", "measure both in the Y basis: always disagree"),
                       ("ZI", "look at qubit x on its own: perfectly random")]:
    print(f"<{pauli}> = {round(b2.inspect.expectation(pauli), 12):5}   ({reading})")

Each qubit individually is completely random ($\langle ZI \rangle = 0$), yet the pair
agrees perfectly in the $Z$ basis *and* perfectly in the $X$ basis — two questions
that, for a single qubit, cannot both have definite answers. Classical correlation
between two random bits can be arranged to survive one such question. Surviving all
of them at once is what notebook 03 turns into an inequality with a number attached.

### Three qubits: GHZ

Extend the recipe: `H` on the first qubit, then a chain of CNOTs. With three qubits
this gives the **GHZ state** (Greenberger–Horne–Zeilinger),
$\tfrac{1}{\sqrt2}(|000\rangle + |111\rangle)$ — all three qubits agreeing, always.

We use a `Register` here, which is just an ordered group of qubit handles; `reg[0]`
is the most significant bit, matching the state-tensor convention.

In [ ]:
ghz = Circuit(name="ghz", seed=5)
g = ghz.register(3, name="g")
H(g[0])
CNOT(g[0], g[1])
CNOT(g[1], g[2])

print(ghz.inspect.ket())
print("sampling 2000 shots without collapsing:", ghz.inspect.sample(2000))

Only two outcomes ever appear, `000` and `111`, roughly 1000 each. The four other
bit patterns have amplitude exactly zero.

`sample` is the one thing here a real machine could also do — except that a real
machine would have to rerun the whole circuit for each shot, since the first
measurement destroys the state. Sampling repeatedly from an intact state is the
cheat, which is why it lives behind `inspect`.

The correlations are all-or-nothing in a strong sense: measuring *any one* of the
three qubits fixes the other two immediately.

In [ ]:
for seed in (5, 6, 8):
    c = Circuit(seed=seed)
    r = c.register(3, name="g")
    H(r[0])
    CNOT(r[0], r[1])
    CNOT(r[1], r[2])
    first = c.measure(r[0])
    print(f"seed {seed}: measured g0 -> {first}; state is now {c.inspect.ket()}; "
          f"measure_all -> {c.measure_all(r)}")

`measure_all` returns the register's bits as one integer with `reg[0]` as the most
significant bit, so the only two possible readings are $0 = |000\rangle$ and
$7 = |111\rangle$. Never 3, never 5.

Each individual qubit of a GHZ state is again maximally mixed — one full bit of
entropy — and so is each *pair*:

In [ ]:
print("entropy of g0 alone:      ", bits(ghz.inspect.entanglement_entropy([g[0]])), "bits")
print("entropy of {g0, g1}:      ", bits(ghz.inspect.entanglement_entropy([g[0], g[1]])), "bits")
print("mutual information g0:g1: ", bits(ghz.inspect.mutual_information([g[0]], [g[1]])), "bits")
print("\nreduced density matrix of {g0, g1}:\n",
      np.round(ghz.inspect.reduced_density_matrix([g[0], g[1]]).real, 3))

That last matrix is worth a second look. Two qubits, so a $4 \times 4$ density matrix
indexed by $|00\rangle, |01\rangle, |10\rangle, |11\rangle$; the only nonzero entries
are $0.5$ in the top-left and $0.5$ in the bottom-right corners. Diagonal, no
coherences: `g0` and `g1` on their own are a *classical* coin flip, correlated but
not entangled with each other. All the quantum coherence of the GHZ state lives in
the three-way relationship, and any attempt to look at only part of it finds
ordinary classical correlation. Tracing out even one qubit destroys it.

Note also that the mutual information between `g0` and `g1` is 1 bit here, not the 2
bits a Bell pair had — a compact statement of the same fact.

## 6. Cuts, and how entanglement grows

Everything above measured entanglement across one particular split of the system.
For more than two qubits there are many splits, and the entropy depends on which one
you take. Choosing a subset is choosing a **cut**: the qubits you keep on one side,
everything else on the other.

Build a four-qubit circuit containing two independent Bell pairs — `q0` with `q1`,
and `q2` with `q3` — and ask about several different cuts.

In [ ]:
cuts = Circuit(name="two pairs", seed=3)
q0, q1, q2, q3 = cuts.alloc_many(4)
H(q0)                    # first Bell pair ...
CNOT(q0, q1)
H(q2)                    # ... and a second one, entirely independent of it
CNOT(q2, q3)

print(cuts.inspect.ket())
print()
for subset, label in [([q0], "{q0}"), ([q0, q1], "{q0,q1}"),
                      ([q0, q2], "{q0,q2}"), ([q0, q1, q2], "{q0,q1,q2}")]:
    s = bits(cuts.inspect.entanglement_entropy(subset))
    product = cuts.inspect.is_product(subset)
    print(f"{label:>12}:  entropy {s:5.3f} bits   is_product={product}")

Four cuts, four different answers, and each one is telling you something specific
about the structure of the state:

- **`{q0}`** — 1 bit. `q0` is maximally entangled with the rest, as half a Bell pair
  should be.
- **`{q0,q1}`** — 0 bits, and `is_product` is `True`. The first pair as a unit is
  completely independent of the second pair. This is the honest sense in which the
  state "is made of two parts": not two qubits, but two *pairs*. Entanglement does
  not prevent a system from decomposing — it just decides where the seams are.
- **`{q0,q2}`** — 2 bits. One qubit from each pair, and each is separately maximally
  entangled with its partner on the other side of the cut, so the entropies add.
- **`{q0,q1,q2}`** — 1 bit, since the only thing crossing this cut is `q2`'s link to
  `q3`.

Notice that `is_product` is not a property of a state so much as of a state *and a
cut*. Asking "is this state entangled?" without saying entangled across what is an
incomplete question.

### Entanglement grows

A last observation, and a preview of something later notebooks lean on heavily.
Start from $|000000\rangle$ — a product state, zero entanglement across every cut —
and apply gates at random. Alternate single-qubit rotations `Ry(q, theta=...)` with
`CNOT`s between random pairs, and after each gate measure the entropy across the cut
that puts the first three qubits on one side and the last three on the other.

In [ ]:
rng = np.random.default_rng(20)          # a separate rng, just for choosing gates
scramble = Circuit(name="scramble", seed=7)
qs = scramble.alloc_many(6)
left = list(qs[:3])                      # the cut: first three qubits vs last three

trace = [scramble.inspect.entanglement_entropy(left)]
for step in range(60):
    if step % 2 == 0:
        k = int(rng.integers(6))
        Ry(qs[k], theta=float(rng.uniform(0, 2 * np.pi)))
    else:
        # choice(..., replace=False) picks two *distinct* qubits; a CNOT from a qubit
        # to itself is not a legal operation and qsim would refuse it.
        i, j = (int(v) for v in rng.choice(6, size=2, replace=False))
        CNOT(qs[i], qs[j])
    trace.append(scramble.inspect.entanglement_entropy(left))

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(trace, linewidth=1.6)
ax.axhline(3.0, color="gray", linestyle="--", linewidth=1)
ax.text(1, 3.05, "maximum for a 3-vs-3 cut: 3 bits", color="gray", fontsize=9)
ax.set_xlabel("gates applied")
ax.set_ylabel("entanglement entropy (bits)")
ax.set_title("entanglement across the middle of a random 6-qubit circuit")
ax.set_ylim(-0.1, 3.4)
print("final entropy:", round(trace[-1], 3), "bits")

The curve starts at zero, climbs in steps — each CNOT that straddles the cut can add
entanglement, each one that does not is flat — and then levels off somewhere around
2 bits, in the neighbourhood of the 3-bit ceiling for this cut. It never comes back
down for long. Random unitary dynamics *manufactures* entanglement; going back to a
product state requires a conspiracy of gates that undoes exactly what was done.

That asymmetry is the seed of a much bigger story. Every real system is coupled to
its surroundings, and that coupling is a circuit of exactly this kind, running
constantly and with no one arranging its gates. Entanglement between a system and
its environment spreads immediately and irreversibly, and — as notebook 05 shows —
what a local observer sees once that has happened is a state with the coherences
scrubbed out of it: a classical mixture. The classical world does not sit underneath
quantum mechanics as a foundation; it is what quantum mechanics looks like after
entanglement with an environment you are not tracking. The graph above is the first
frame of that process.

## 7. Why a `Qubit` has no `.state`

We can now close a loop that notebook 01 left open. The obvious API for a simulator
would give each qubit a `.state` attribute — two amplitudes, easily printed. qsim
deliberately does not, and by this point in the notebook the reason should sound less
like a restriction and more like a description.

A `Qubit` in qsim is a **handle**: it stores a stable id and asks its `Circuit` which
axis of the shared state tensor that id currently names. There is exactly one state
in the system and the circuit owns it. Give a qubit a `.state` property and you
immediately have to decide what it returns for half of a Bell pair — and the honest
answer, established in section 3, is that no state vector exists. The property would
have to invent one, or return $|{+}\rangle$ (wrong), or throw. So it does not exist,
and the correct question — "what does this subsystem look like on its own?" — is
asked with `inspect.reduced_density_matrix([q])`, which can answer it truthfully
because a density matrix is a big enough language.

The same reasoning shows up more sharply if you try to copy a handle.

In [ ]:
c, x, y = fresh_bell(1)

try:
    twin = copy.copy(x)
except NoCloningError as err:
    print("NoCloningError:", err)

print()
print("does a Qubit have a .state attribute? ", hasattr(x, "state"))
print("what it does have instead:", repr(x))

The error message is the point:

> *cannot copy q0. This is the no-cloning theorem: there is no physical operation
> that copies an unknown quantum state, because copying is not a linear map and
> quantum evolution is linear. Copying the handle would suggest otherwise. If two
> variables should refer to the same qubit, plain assignment does that; if you want a
> second qubit prepared the same way, allocate one and repeat the gates that prepared
> this one.*

The **no-cloning theorem** is a two-line proof. Suppose some unitary $U$ copied
states: $U(|\psi\rangle \otimes |0\rangle) = |\psi\rangle \otimes |\psi\rangle$ for
every $|\psi\rangle$. Apply it to $|0\rangle$ and to $|1\rangle$, then to
$\tfrac{1}{\sqrt2}(|0\rangle + |1\rangle)$. Linearity forces the last result to be
$\tfrac{1}{\sqrt2}(|00\rangle + |11\rangle)$ — a Bell state — but the copying rule
demands $|{+}\rangle \otimes |{+}\rangle$, which section 2 proved is a different
state. The only maps that copy are non-linear ones, and quantum evolution is linear.

So `copy.copy(qubit)` raises rather than quietly handing back a second handle that
looks like a duplicate. And the same principle appeared earlier without the word
attached: passing the same qubit twice to a two-qubit gate is refused too, because a
qubit controlling an operation on itself would have to read its own value.

In [ ]:
try:
    CNOT(x, x)
except NoCloningError as err:
    print("NoCloningError:", err)

Both refusals come from the same place. The library's object model is not being
fussy; it is declining to offer operations that do not exist. A handle names an axis
of a shared tensor, entangled qubits have no individual states, and an API that
pretended otherwise would teach you something false every time you used it.

## What you now know

- A **product state** factors as $|\psi\rangle \otimes |\phi\rangle$. Its parts each
  have a state of their own, a Bloch vector of length 1, and zero entanglement
  entropy. $H \otimes H\,|00\rangle$ is one.
- A state is **entangled** exactly when no such factorization exists. For the Bell
  state $\tfrac{1}{\sqrt2}(|00\rangle + |11\rangle)$, matching coefficients gives
  $\alpha\gamma = \beta\delta = \tfrac{1}{\sqrt2}$ and $\alpha\delta = \beta\gamma = 0$,
  which has no solution. Equivalently: the matricized state has rank 2 rather than 1.
- A **density matrix** describes both pure states and statistical mixtures. Its
  diagonal holds probabilities; its off-diagonal **coherences** are what remains of
  superposition. The **partial trace** ($\rho_A = MM^\dagger$) averages away the
  qubits you chose not to look at.
- One half of a Bell pair traces out to the **maximally mixed state** $\tfrac12 I$ —
  a Bloch vector of length zero at the centre of the sphere. Maximal knowledge of the
  whole, minimal knowledge of every part: a combination classical physics has no room
  for.
- The **Schmidt decomposition** writes any bipartite pure state as a single sum
  $\sum_i s_i |a_i\rangle|b_i\rangle$; the $s_i$ are singular values, the $s_i^2$ are
  a probability distribution, and their Shannon entropy is the **entanglement
  entropy**. It is $0$ for product states and exactly $1$ bit for a Bell pair, with a
  continuum in between.
- Measuring one half projects the **joint** state, so the other half becomes definite
  at once — while its reduced density matrix, averaged over outcomes, is unchanged, so
  nothing is signalled.
- The **GHZ** state $\tfrac{1}{\sqrt2}(|000\rangle + |111\rangle)$ correlates three
  qubits all-or-nothing, and tracing out any one of them leaves ordinary classical
  correlation behind.
- Entanglement is measured **across a cut**, and random circuits drive it up quickly
  and keep it there.
- A `Qubit` has no `.state` and cannot be copied, because an entangled qubit has no
  state of its own to report and no unitary can duplicate an unknown one.

Useful `inspect` methods from this notebook: `reduced_density_matrix`,
`schmidt_spectrum`, `entanglement_entropy`, `is_product`, `mutual_information`,
`bloch_vector`, `expectation`, `sample`.

## Next: notebook 03

One question has been left standing since section 5. Two boxes containing gloves are
perfectly correlated, and there is nothing mysterious about them: the correlation was
put there when the boxes were packed. Why is a Bell pair not just a very small pair
of gloves?

`03-bell-tests-teleportation.ipynb` settles it. Bell's insight was that the glove story — any
story in which each qubit carries pre-existing answers to every question that might
be asked — puts a hard numerical ceiling on how strongly the results can correlate
when the two sides choose their measurement bases independently. The **CHSH**
quantity cannot exceed $2$ under any such story. A Bell pair reaches $2\sqrt2 \approx 2.83$,
and we will build the circuit and watch it happen. Entanglement is not classical
correlation, and that notebook is where the difference stops being a matter of
interpretation and becomes a number you can measure.